# 07 Model Family Comparison

## Overview

Use this notebook before the family specific fine tune notebooks.
It compares the teaching checkpoint for each family, shows access and hardware notes side by side, and renders one sample conversation through each tokenizer so you can see prompt and tokenization differences.

## Prerequisites

* Install the notebook dependencies in the next cell.
* Log in to Hugging Face first if you want tokenizer access for gated families such as Gemma or Llama.
* Expect some families to fail gracefully if you do not have access yet.


In [ ]:
%pip install -q transformers==4.48.2 huggingface_hub==0.22.2


In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

from transformers import AutoTokenizer
from utils.model_families import MODEL_FAMILIES, get_model_family_rows


## Side By Side Family View

This cell prints the shared registry that drives the new family notebooks.


In [ ]:
rows = get_model_family_rows()
rows


## Prompt Rendering And Token Counts

This uses the real tokenizer for each family when access is available.
That matters because prompt wrappers differ across model families.


In [ ]:
sample_messages = [
    {"role": "system", "content": "You are a concise machine learning tutor."},
    {"role": "user", "content": "Explain what LoRA changes during fine tuning in two short sentences."},
]

comparison_results = []

for profile in MODEL_FAMILIES:
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            profile.comparison_model_id,
            trust_remote_code=profile.trust_remote_code,
        )
        if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
            rendered = tokenizer.apply_chat_template(
                sample_messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            token_count = len(tokenizer(rendered)["input_ids"])
        else:
            rendered = "Tokenizer chat template not available for this checkpoint."
            token_count = "fallback"
        comparison_results.append(
            {
                "family": profile.family,
                "comparison_model_id": profile.comparison_model_id,
                "token_count": token_count,
                "rendered_prompt_preview": rendered[:500],
            }
        )
    except Exception as exc:
        comparison_results.append(
            {
                "family": profile.family,
                "comparison_model_id": profile.comparison_model_id,
                "token_count": "unavailable",
                "rendered_prompt_preview": f"Could not render tokenizer template: {exc}",
            }
        )

comparison_results


## What To Look For

* Families do not tokenize the same rendered prompt the same way.
* Gated families will force you to solve access before training.
* Bigger families need more than just more VRAM. They also slow iteration and debugging.
* Kimi K2 belongs in compare mode for most learners. The full example track uses a lighter Moonshot family checkpoint instead.
